In [1]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [2]:
from scale_rl.common.wandb_utils import *

In [3]:
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_RANDOM_SCORE, HB_SUCCESS_SCORE

def replace_underbar_to_hypen(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_underbar_to_hypen_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

HB_LOCOMOTION_NOHAND = replace_underbar_to_hypen(HB_LOCOMOTION_NOHAND)

HB_SUCCESS_SCORE = replace_underbar_to_hypen_dict(HB_SUCCESS_SCORE)
HB_RANDOM_SCORE = replace_underbar_to_hypen_dict(HB_RANDOM_SCORE)

/home/nas4_user/youngdolee/anaconda3/envs/rl_playground/lib/python3.9/site-packages/glfw/__init__.py:916: GLFWError: (65544) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


In [4]:
entity = 'draftrec'
project_name = 'CQN_AS'

run_exp_names_to_analysis_exp_names = {
    "h1-walk-v0": "CQN_AS",
    "h1-stand-v0": "CQN_AS",
    "h1-run-v0": "CQN_AS",
    "h1-reach-v0": "CQN_AS",
    "h1-hurdle-v0": "CQN_AS",
    "h1-crawl-v0": "CQN_AS",
    "h1-maze-v0": "CQN_AS",
    "h1-sit_simple-v0": "CQN_AS",
    "h1-sit_hard-v0": "CQN_AS",
    "h1-balance_simple-v0": "CQN_AS",
    "h1-balance_hard-v0": "CQN_AS",
    "h1-stair-v0": "CQN_AS",
    "h1-slide-v0": "CQN_AS",
    "h1-pole-v0": "CQN_AS",
}
run_exp_names = list(run_exp_names_to_analysis_exp_names.keys())
metrics = ['eval/episode_reward']

In [5]:
def _convert_wandb_df_to_eval_df(runs_df, metrics: List, n_samples=100000):
    """
    Collect the history of specified metrics from wandb runs.

    Parameters:
    runs_df: pandas DataFrame
        DataFrame containing the wandb runs.
    metrics: list
        List of metric names to collect history for.
    n_samples: int, optional
        Number of samples to fetch from the run history. Default is 100000.

    Returns:
    eval_df: pandas DataFrame with following columns
        exp_name / env_name / seed / metric / env_step / value
    """
    records = []

    for idx in tqdm(range(len(runs_df))):
        run_data = runs_df.iloc[idx]
        exp_name = run_data["exp_name"]
        config = run_data["config"]
        if "env_name" in config:
            env_name = config["env_name"]
        elif "env" in config:
            env_name = config["env"]["env_name"]
        else:
            raise ValueError
        seed = config["seed"]

        run = run_data["run"]
        history = run.history(samples=n_samples)
        steps = history["eval/step"]

        for metric in metrics:
            if metric in history.columns:
                metric_history = history[metric].dropna()
                for idx, val in metric_history.items():
                    env_step = steps[idx]

                    records.append(
                        {
                            "exp_name": exp_name,
                            "env_name": env_name,
                            "seed": seed,
                            "metric": metric,
                            "env_step": env_step,
                            "value": val,
                        }
                    )

    return pd.DataFrame(records)

In [6]:
runs = collect_runs(entity=entity, project_name=project_name)
for run in tqdm(runs):
    run.config["group_name"] = 'CQN_AS'
    run.config['seed'] = int(run.config['seed'])
    run.config["env_name"] = run.config['task_name']
    run.config["exp_name"] = run.config['task_name']

  0%|          | 0/70 [00:00<?, ?it/s]

In [7]:
filtered_runs = filter_runs(runs, exp_names = run_exp_names)
wandb_df = convert_runs_to_dataframe(
    runs = filtered_runs, 
    run_exp_name_to_analysis_exp_name=run_exp_names_to_analysis_exp_names
)
wandb_df = wandb_df[wandb_df.apply(lambda row: 'finished' in str(row['run']), axis=1)]
eval_df = _convert_wandb_df_to_eval_df(wandb_df, metrics)
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
eval_df


  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

  0%|          | 0/70 [00:00<?, ?it/s]

,exp_name,env_name,seed,metric,env_step,value
0,CQN_AS,h1-sit-simple-v0,2000,eval/episode_reward,50067.0,138.113117
1,CQN_AS,h1-sit-simple-v0,2000,eval/episode_reward,100387.0,186.576757
2,CQN_AS,h1-sit-simple-v0,2000,eval/episode_reward,150222.0,150.791210
3,CQN_AS,h1-sit-simple-v0,2000,eval/episode_reward,200005.0,233.846568
4,CQN_AS,h1-sit-simple-v0,2000,eval/episode_reward,250731.0,458.634406
...,...,...,...,...,...,...
1325,CQN_AS,h1-maze-v0,3000,eval/episode_reward,750128.0,119.418723
1326,CQN_AS,h1-maze-v0,3000,eval/episode_reward,800037.0,114.583109
1327,CQN_AS,h1-maze-v0,3000,eval/episode_reward,850006.0,115.824838
1328,CQN_AS,h1-maze-v0,3000,eval/episode_reward,900077.0,116.124173


In [8]:
eval_df['metric'] = eval_df['metric'].map({'eval/episode_reward': 'avg_return'})
eval_df['env_step'] = eval_df['env_step'] // 10000 * 10000
eval_df['env_step'] = eval_df['env_step'].astype(int)
eval_df

,exp_name,env_name,seed,metric,env_step,value
0,CQN_AS,h1-sit-simple-v0,2000,avg_return,50000,138.113117
1,CQN_AS,h1-sit-simple-v0,2000,avg_return,100000,186.576757
2,CQN_AS,h1-sit-simple-v0,2000,avg_return,150000,150.791210
3,CQN_AS,h1-sit-simple-v0,2000,avg_return,200000,233.846568
4,CQN_AS,h1-sit-simple-v0,2000,avg_return,250000,458.634406
...,...,...,...,...,...,...
1325,CQN_AS,h1-maze-v0,3000,avg_return,750000,119.418723
1326,CQN_AS,h1-maze-v0,3000,avg_return,800000,114.583109
1327,CQN_AS,h1-maze-v0,3000,avg_return,850000,115.824838
1328,CQN_AS,h1-maze-v0,3000,avg_return,900000,116.124173


In [9]:
eval_df.to_csv("../results/0110_cqn_as_hbench_1m.csv")

In [10]:
HB_STEPS = 1e6 # 1M

In [12]:
_eval_df = normalize_score_with_random_and_base_score(eval_df, HB_RANDOM_SCORE, HB_SUCCESS_SCORE) # eval_df
overall_df = []
for env_name in HB_LOCOMOTION_NOHAND:
    env_df = _eval_df[_eval_df["env_name"] == env_name]
    if len(env_df) == 0:
        continue
    # assert max(env_df["env_step"]) == HB_STEPS
    env_df = env_df[env_df["env_step"] == max(env_df["env_step"])]
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean, 3)} ± {round(1.96 * std_err, 3)}")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

h1-walk-v0 - number of seeds: 5
0.077 ± 0.026
h1-stand-v0 - number of seeds: 5
0.543 ± 0.286
h1-run-v0 - number of seeds: 5
0.029 ± 0.018
h1-reach-v0 - number of seeds: 5
0.135 ± 0.053
h1-hurdle-v0 - number of seeds: 5
0.03 ± 0.015
h1-crawl-v0 - number of seeds: 5
0.684 ± 0.048
h1-maze-v0 - number of seeds: 5
0.019 ± 0.014
h1-sit-simple-v0 - number of seeds: 5
0.922 ± 0.243
h1-sit-hard-v0 - number of seeds: 5
0.292 ± 0.103
h1-balance-simple-v0 - number of seeds: 5
0.069 ± 0.008
h1-balance-hard-v0 - number of seeds: 5
0.048 ± 0.003
h1-stair-v0 - number of seeds: 5
0.063 ± 0.011
h1-slide-v0 - number of seeds: 5
0.069 ± 0.005
h1-pole-v0 - number of seeds: 5
0.121 ± 0.013
Overall: 0.221 ± 0.07
